In [0]:
from pyspark.sql.functions import col, regexp_replace, to_date, date_format, sum, when

# 1. Load expenses.csv from your Unity Catalog Volume
expenses_df = spark.read.csv(
    "/Volumes/workspace/default/expense_files/expenses.csv",
    header=True,
    inferSchema=True
)

# Clean amount ($ and commas) and extract year-month
expenses_df = expenses_df.withColumn(
    "amount", regexp_replace(col("amount"), r"[\$,]", "").cast("double")
).withColumn(
    "date", to_date(col("date"), "yyyy-MM-dd")
).withColumn(
    "month", date_format(col("date"), "yyyy-MM")
)

# 2. Create the Week 1 users DataFrame directly in PySpark (no separate CSV needed)
users_data = [
    (1, "jersil", "jersil@example.com", 300.00),
    (2, "alex", "alex@example.com", 1300.00)
]
users_df = spark.createDataFrame(users_data, ["user_id", "username", "email", "monthly_budget"])

print("Users Table:")
users_df.show()

print("Cleaned Expenses Table:")
expenses_df.show()

Users Table:
+-------+--------+------------------+--------------+
|user_id|username|             email|monthly_budget|
+-------+--------+------------------+--------------+
|      1|  jersil|jersil@example.com|         300.0|
|      2|    alex|  alex@example.com|        1300.0|
+-------+--------+------------------+--------------+

Cleaned Expenses Table:
+----------+-------+----------+-------------+------+--------------------+-------+
|expense_id|user_id|      date|     category|amount|         description|  month|
+----------+-------+----------+-------------+------+--------------------+-------+
|       101|      1|2026-01-05|    Groceries| 125.5|Weekly supermarke...|2026-01|
|       102|      1|2026-01-12|    Utilities|  85.0|    Electricity bill|2026-01|
|       103|      2|2026-01-15|Entertainment|  45.0|       Movie tickets|2026-01|
|       104|      1|2026-01-20|    Groceries|  64.2|        Local market|2026-01|
|       105|      2|2026-01-28|         Rent|1200.0|  January house re

In [0]:
combined_df = expenses_df.join(users_df, on="user_id", how="inner")

print("Combined User and Expense Data:")
combined_df.select("expense_id", "user_id", "username", "month", "category", "amount", "monthly_budget").show()

Combined User and Expense Data:
+----------+-------+--------+-------+-------------+------+--------------+
|expense_id|user_id|username|  month|     category|amount|monthly_budget|
+----------+-------+--------+-------+-------------+------+--------------+
|       101|      1|  jersil|2026-01|    Groceries| 125.5|         300.0|
|       102|      1|  jersil|2026-01|    Utilities|  85.0|         300.0|
|       103|      2|    alex|2026-01|Entertainment|  45.0|        1300.0|
|       104|      1|  jersil|2026-01|    Groceries|  64.2|         300.0|
|       105|      2|    alex|2026-01|         Rent|1200.0|        1300.0|
|       106|      1|  jersil|2026-02|    Groceries|140.75|         300.0|
|       107|      2|    alex|2026-02|    Utilities|  92.3|        1300.0|
|       108|      1|  jersil|2026-02|Entertainment| 110.0|         300.0|
|       109|      2|    alex|2026-02|         Rent|1200.0|        1300.0|
|       110|      1|  jersil|2026-03|   Healthcare|  55.4|         300.0|
+-----

In [0]:
monthly_summary = combined_df.groupBy("user_id", "username", "monthly_budget", "month").agg(
    sum("amount").alias("total_monthly_spend")
)

final_report = monthly_summary.withColumn(
    "savings",
    col("monthly_budget") - col("total_monthly_spend")
).withColumn(
    "alert_status",
    when(col("total_monthly_spend") > col("monthly_budget"), "ALERT: Budget Exceeded")
    .when(col("total_monthly_spend") >= (col("monthly_budget") * 0.8), "WARNING: Near Budget Limit")
    .otherwise("OK: Within Budget")
).orderBy("month", "user_id")

print("Final Monthly Spend, Savings & Alerts Report:")
final_report.show(truncate=False)

Final Monthly Spend, Savings & Alerts Report:
+-------+--------+--------------+-------+-------------------+------------------+--------------------------+
|user_id|username|monthly_budget|month  |total_monthly_spend|savings           |alert_status              |
+-------+--------+--------------+-------+-------------------+------------------+--------------------------+
|1      |jersil  |300.0         |2026-01|274.7              |25.30000000000001 |WARNING: Near Budget Limit|
|2      |alex    |1300.0        |2026-01|1245.0             |55.0              |WARNING: Near Budget Limit|
|1      |jersil  |300.0         |2026-02|250.75             |49.25             |WARNING: Near Budget Limit|
|2      |alex    |1300.0        |2026-02|1292.3             |7.7000000000000455|WARNING: Near Budget Limit|
|1      |jersil  |300.0         |2026-03|55.4               |244.6             |OK: Within Budget         |
+-------+--------+--------------+-------+-------------------+------------------+----------

In [0]:
# 1. Save as Delta Table in workspace.default
final_report.write.format("delta").mode("overwrite").saveAsTable("workspace.default.monthly_expense_summary")
print("Saved Delta Table: workspace.default.monthly_expense_summary")

# 2. Save as CSV in your volume for easy download
final_report.toPandas().to_csv("/Volumes/workspace/default/expense_files/final_monthly_expense_report.csv", index=False)
print("Saved CSV: /Volumes/workspace/default/expense_files/final_monthly_expense_report.csv")

Saved Delta Table: workspace.default.monthly_expense_summary
Saved CSV: /Volumes/workspace/default/expense_files/final_monthly_expense_report.csv


In [0]:
%sql
SELECT * FROM workspace.default.monthly_expense_summary
ORDER BY month, user_id;

user_id,username,monthly_budget,month,total_monthly_spend,savings,alert_status
1,jersil,300.0,2026-01,274.7,25.30000000000001,WARNING: Near Budget Limit
2,alex,1300.0,2026-01,1245.0,55.0,WARNING: Near Budget Limit
1,jersil,300.0,2026-02,250.75,49.25,WARNING: Near Budget Limit
2,alex,1300.0,2026-02,1292.3,7.7000000000000455,WARNING: Near Budget Limit
1,jersil,300.0,2026-03,55.4,244.6,OK: Within Budget
